In [ ]:
import pandas as pd




In [ ]:
columnas_base = ["engine_id", "time_cycle", "op_setting_1", "op_setting_2", "op_setting_3"]

# 2. Generamos los 21 sensores (del 1 al 21). El range(1, 22) llega hasta el 21.
columnas_sensores = [f"sensor_{i}" for i in range(1, 22)]

# 3. Concatenamos ambas listas (en Python, sumar dos listas las une)
todas_las_columnas = columnas_base + columnas_sensores

# 4. Leemos el archivo. El parámetro 'names' hace el trabajo sucio por nosotros.
df = pd.read_csv("data/train_FD001.txt", sep=r"\s+", names=todas_las_columnas)



In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()


In [ ]:
df.isnull().sum()

In [ ]:
df = df.drop(columns=["op_setting_3", "sensor_18","sensor_19"])

In [ ]:
df.shape

flujo algorítmico para agrupar, renombrar, cruzar y clasificar. calcular el La columna RUL , la NASA no nos entregó una columna que diga "Fallo". Necesitamos construirla mediante ingeniería de características. Vamos a calcular el RUL (Remaining Useful Life o Vida Útil Restante).Construir la Variable Predictiva.

In [ ]:
ciclos_maximos = df.groupby('engine_id')['time_cycle'].max().reset_index()
ciclos_maximos.columns = ['engine_id', 'max_cycle']
df = pd.merge(df, ciclos_maximos, how='left', on='engine_id')
df['RUL'] = df['max_cycle'] - df['time_cycle']
df['label'] = (df['RUL'] <= 30).astype(int)


In [ ]:
df[['engine_id', 'time_cycle', 'max_cycle', 'RUL', 'label']].head(10)

In [ ]:
df[df['engine_id'] == 1].tail(10)

Fase 2: El Cerebro (Entrenamiento del Modelo).

Antes de inyectar estos datos a un algoritmo de Machine Learning, necesitamos aplicar un principio crítico de MLOps en nuestra capa de datos para evitar el Data Leakage (Fuga de Datos). Si al modelo le pasamos columnas como el ciclo máximo o el RUL, este no aprenderá a leer los sensores, sino que simplemente hará la resta matemática y hará "trampa" en el examen.

Vamos a separar nuestros datos en variables independientes (telemetría pura) y nuestra variable dependiente (la respuesta)

In [ ]:
x = df.drop(['engine_id','time_cycle','max_cycle','RUL','label'], axis=1)
y = df['label']

Importación del Módulo: Importa la herramienta de división usando

In [ ]:
from sklearn.model_selection import train_test_split

División (Train/Test Split): Utiliza esta función para dividir X e y en datos de entrenamiento y prueba. Asigna el resultado a cuatro variables

In [ ]:

# Dividir los datos en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)


In [ ]:
print( "Datos de entrenamiento X" ,X_train.shape)
print( "Datos de entrenamiento y" ,y_train.shape)

In [ ]:
from sklearn.ensemble import RandomForestClassifier 


